# Search for Examinations and Medications
Extract relevant examinations and medications for the CKD project dataset from the OMOP table

Isobel Weinberg, Aug 2025

In [ ]:
import sys
sys.path.append('..')
from credentials import *

from elasticsearch_utils import *

import duckdb
import pandas as pd
import numpy as np
import re

data_path = "../data/"
raw_data_path = data_path+'raw_data/'

## Load Patients of Interest

In [ ]:
cols = ['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "chronic_kidney_disease_refined_inclusion_patients.csv"))[cols]

## List of outputs

### Medications: 
- RAAS blockers (ACEi/ARB)
- SGLT2 inhibitors 
- Regular use of NSAIDs
- Diuretics
- Nephrotoxic drugs e.g. lithium, immunosupressants
- Antihypertensives
- Diabetic medications
- Statins
- Anti-platelets
- Polypharmacy (number of repeat medications)

## Medications

In [ ]:
with duckdb.connect() as conn:
    omop_medication_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                              SELECT DISTINCT de.master_person_id
                                     , de.master_visit_occurrence_id
                                     , de.drug_exposure_start_date
                                     , de.drug_source_value_name AS drug
                                     , CASE WHEN UPPER(de.drug_source_value_name) LIKE '%AMLODIPINE%' THEN 'Amlodipine'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%FELODIPINE%' THEN 'Felodipine'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%NIFEDIPINE%' THEN 'Nifedipine'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%LERCANIDIPINE%' THEN 'Lercanidipine'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%VERAPAMIL%' THEN 'Verapamil'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%RAMIPRIL%' THEN 'Ramipril'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%ENALAPRIL%' THEN 'Enalapril'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%LISINOPRIL%' THEN 'Lisinopril'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%PERINDOPRIL%' THEN 'Perindopril'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%TRANDOLAPRIL%' THEN 'Trandolapril'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%CANDESARTAN%' THEN 'Candesartan'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%IRBESARTAN%' THEN 'Irbesartan'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%LOSARTAN%' THEN 'Losartan'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%OLMESARTAN%' THEN 'Olmesartan'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%TELMISARTAN%' THEN 'Telmisartan'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%VALSARTAN%' THEN 'Valsartan'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%DAPAGLIFLOZIN%' THEN 'Dapagliflozin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%EMPAGLIFLOZIN%' THEN 'Empagliflozin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%CANAGLIFLOZIN%' THEN 'Canagliflozin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%ISOSORBIDE%' THEN 'Isosorbide'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%ENTRESTO%' THEN 'Entresto'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%SACUBITRIL%' THEN 'Sacubitril'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%BISOPROLOL%' THEN 'Bisoprolol'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%ATENOLOL%' THEN 'Atenolol'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%CARVEDILOL%' THEN 'Carvedilol'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%NEBIVOLOL%' THEN 'Nebivolol'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%DOXAZOSIN%' THEN 'Doxazosin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%CLOPIDOGREL%' THEN 'Clopidogrel'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%ATORVASTATIN%' THEN 'Atorvastatin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%SIMVASTATIN%' THEN 'Simvastatin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%PRAVASTATIN%' THEN 'Pravastatin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%ROSUVASTATIN%' THEN 'Rosuvastatin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%FLUVASTATIN%' THEN 'Fluvastatin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%EZETIMIB%' THEN 'Ezetimibe'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%HYDRALAZINE%' THEN 'Hydralazine'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%MINOXIDIL%' THEN 'Minoxidil'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%MOXONIDINE%' THEN 'Moxonidine'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%SPIRONOLACTONE%' THEN 'Spironolactone'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%EPLERENONE%' THEN 'Eplerenone'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%FINERENONE%' THEN 'Finerenone'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%INSULIN%' THEN 'Insulin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%METFORMIN%' THEN 'Metformin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%GLICLAZIDE%' THEN 'Gliclazide'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%EXENATIDE%' THEN 'Exenatide'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%LIRAGLUTIDE%' THEN 'Liraglutide'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%DULAGLUTIDE%' THEN 'Dulaglutide'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%SEMAGLUTIDE%' THEN 'Semaglutide'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%LINAGLIPTIN%' THEN 'Linagliptin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%SITAGLIPTIN%' THEN 'Sitagliptin'
                                            WHEN UPPER(de.drug_source_value_name) LIKE '%MONONITRATE%' THEN 'Mononitrate'
                                            ELSE '' END AS drug_name
                                     , de.drug_status_source_value AS status
                                     , de.drug_location_source_value AS location
                                     , de.dose_quantity_source_value AS dose
                                     , de.dose_unit_source_value AS dose_unit
                                     , de.route_concept_name AS roi
                              FROM ext_drug_exposure AS de
                                  INNER JOIN inclusion_patients_df AS ip
                                      ON de.master_person_id = ip.master_person_id
                              WHERE UPPER(de.drug_source_value_name) LIKE '%AMLODIPINE%' OR UPPER(de.drug_source_value_name) LIKE '%FELODIPINE%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%NIFEDIPINE%' OR UPPER(de.drug_source_value_name) LIKE '%LERCANIDIPINE%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%VERAPAMIL%' OR UPPER(de.drug_source_value_name) LIKE '%RAMIPRIL%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%ENALAPRIL%' OR UPPER(de.drug_source_value_name) LIKE '%LISINOPRIL%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%PERINDOPRIL%' OR UPPER(de.drug_source_value_name) LIKE '%TRANDOLAPRIL%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%CANDESARTAN%' OR UPPER(de.drug_source_value_name) LIKE '%IRBESARTAN%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%LOSARTAN%' OR UPPER(de.drug_source_value_name) LIKE '%OLMESARTAN%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%TELMISARTAN%' OR UPPER(de.drug_source_value_name) LIKE '%VALSARTAN%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%DAPAGLIFLOZIN%' OR UPPER(de.drug_source_value_name) LIKE '%EMPAGLIFLOZIN%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%CANAGLIFLOZIN%' OR UPPER(de.drug_source_value_name) LIKE '%ISOSORBIDE%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%ENTRESTO%' OR UPPER(de.drug_source_value_name) LIKE '%SACUBITRIL%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%BISOPROLOL%' OR UPPER(de.drug_source_value_name) LIKE '%ATENOLOL%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%CARVEDILOL%' OR UPPER(de.drug_source_value_name) LIKE '%NEBIVOLOL%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%DOXAZOSIN%' OR UPPER(de.drug_source_value_name) LIKE '%CLOPIDOGREL%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%ATORVASTATIN%' OR UPPER(de.drug_source_value_name) LIKE '%SIMVASTATIN%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%PRAVASTATIN%' OR UPPER(de.drug_source_value_name) LIKE '%ROSUVASTATIN%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%FLUVASTATIN%' OR UPPER(de.drug_source_value_name) LIKE '%EZETIMIB%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%HYDRALAZINE%' OR UPPER(de.drug_source_value_name) LIKE '%MINOXIDIL%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%MOXONIDINE%' OR UPPER(de.drug_source_value_name) LIKE '%SPIRONOLACTONE%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%EPLERENONE%' OR UPPER(de.drug_source_value_name) LIKE '%FINERENONE%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%INSULIN%' OR UPPER(de.drug_source_value_name) LIKE '%METFORMIN%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%GLICLAZIDE%' OR UPPER(de.drug_source_value_name) LIKE '%EXENATIDE%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%LIRAGLUTIDE%' OR UPPER(de.drug_source_value_name) LIKE '%DULAGLUTIDE%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%SEMAGLUTIDE%' OR UPPER(de.drug_source_value_name) LIKE '%LINAGLIPTIN%' OR
                                    UPPER(de.drug_source_value_name) LIKE '%SITAGLIPTIN%' OR UPPER(de.drug_source_value_name) LIKE '%MONONITRATE%'
                              ORDER BY de.master_person_id, de.drug_exposure_start_date;""").df()

In [ ]:
ace_inhibitors = [ "Captopril", "Enalapril", "Lisinopril", "Perindopril", "Ramipril", "Quinapril", "Benazepril", "Cilazapril",
                   "Fosinopril", "Moexipril", "Trandolapril", "Spirapril", "Delapril", "Temocapril", "Zofenopril", "Imidapril"]

arbs = ["Losartan", "Eprosartan", "Valsartan", "Irbesartan", "Candesartan", "Telmisartan", "Olmesartan", "Azilsartan", "Fimasartan"]

entresto = ["Entresto", "Sacubitril"]

ccb = ["Amlodipine", "Felodipine", "Nifedipine", "Lercanidipine", "Verapamil"]

slgt2i = ["Dapagliflosin", "Empagliflozin", "Canagliflozin", "Dapagliflozin"]

beta_blocker = ["Bisoprolol", "Atenolol", "Carvedilol", "Nebivolol"]

statin = ["Atorvastatin", "Simvastatin", "Pravastatin", "Rosuvastatin", "Fluvastatin"]

mra = ["Spironolactone", "Eplerenone", "Finerenone"]

glp1 = ["Exenatide", "Liraglutide", "Dulaglutide", "Semaglutide"]

dpp4 = ["Linagliptin", "Sitagliptin"]

In [ ]:
def drug_classification(x):
    drug_classification = ''

    if x in ace_inhibitors:
        drug_classification = 'Ace Inhibitors'
    elif x in arbs:
        drug_classification = 'ARB'
    elif x in entresto:
        drug_classification = 'Entresto'
    elif x in ccb:
        drug_classification = 'CCB'
    elif x in slgt2i:
        drug_classification = 'SLGT2i'
    elif x in beta_blocker:
        drug_classification = 'Beta Blocker'
    elif x in statin:
        drug_classification = 'Statin'
    elif x in mra:
        drug_classification = 'MRA'
    elif x in glp1:
        drug_classification = 'GLP1'
    elif x in dpp4:
        drug_classification = 'DPP4'

    return drug_classification

In [ ]:
omop_medication_df['drug_classification'] = omop_medication_df['drug_name'].apply(lambda x: drug_classification(x))

## Export Data

In [ ]:
#- Save Results-
file_name = "20251203_medication_search_results.csv"

omop_medication_df.to_csv(f"{raw_data_path}/{file_name}", index=False)
print("✅ Results saved.")